# Notebook KPI

In [52]:
import pandas as pd
import sys
sys.path.append('../src')
from tool import inspector
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1.Import des données

In [53]:
df_b2b_age = pd.read_csv(r"..\data\data_enrichies\df_b2b_age.csv")

In [54]:
#convertion des date au bon format
df_b2b_age['date'] = pd.to_datetime(df_b2b_age['date'])

In [55]:
inspector(df_b2b_age)

######### DEBUT DE L'ANALYSE #########
Le df comprend 687534 lignes et 10 colonnes 

---------------------------------------------------------------
le df contient les nuls suivant 
id_prod           0
date              0
session_id        0
client_id         0
price             0
categ             0
sex               0
birth             0
segment_client    0
age               0
dtype: int64

---------------------------------------------------------------
les types et noms de colonnes sont les suivants
id_prod                   object
date              datetime64[ns]
session_id                object
client_id                 object
price                    float64
categ                      int64
sex                       object
birth                      int64
segment_client            object
age                        int64
dtype: object

---------------------------------------------------------------
le nombre d'uniques dans les colonnes sont les suivants
id_prod             3265
da

In [56]:
df_b2b_age.head()

,id_prod,date,session_id,client_id,price,categ,sex,birth,segment_client,age
0,0_1259,2021-03-01 00:01:07.843138,s_1,c_329,11.99,0,f,1967,B2C,54
1,0_1390,2021-03-01 00:02:26.047414,s_2,c_664,19.37,0,m,1960,B2C,61
2,0_1352,2021-03-01 00:02:38.311413,s_3,c_580,4.50,0,m,1988,B2C,33
3,0_1458,2021-03-01 00:04:54.559692,s_4,c_7912,6.55,0,f,1989,B2C,32
4,0_1358,2021-03-01 00:05:18.801198,s_5,c_2033,16.49,0,f,1956,B2C,65


# 2.Calcul et graphique CA

## 2.1 Chiffre d'affaire et moyenne mobile 

In [57]:
#grouper le CA par jour
ca_journalier = df_b2b_age.groupby(pd.Grouper(key='date', freq='D'))['price'].sum().reset_index()
ca_journalier.rename(columns={"price":"ca"}, inplace=True)
ca_journalier.head()

,date,ca
0,2021-03-01,16565.22
1,2021-03-02,15486.45
2,2021-03-03,15198.69
3,2021-03-04,15196.07
4,2021-03-05,17471.37


In [58]:
#création des CA journalier par categories
ca_journalier_b2b = df_b2b_age[df_b2b_age['segment_client'] == 'B2B'].groupby(pd.Grouper(key='date', freq='D'))['price'].sum().reset_index()
ca_journalier_b2b.rename(columns={"price":"ca"}, inplace=True)
ca_journalier_b2c = df_b2b_age[df_b2b_age['segment_client'] == 'B2C'].groupby(pd.Grouper(key='date', freq='D'))['price'].sum().reset_index()
ca_journalier_b2c.rename(columns={"price":"ca"}, inplace=True)

### Moyenne mobile B2B

In [59]:
#trier par date (inspensable pour les moy mobile)
ca_journalier_b2b = ca_journalier_b2b.sort_values('date').reset_index(drop=True)
#moyenne mobile sur 7 jours
ca_journalier_b2b['mm_7j'] = ca_journalier_b2b['ca'].rolling(window=7).mean()
#moyenne mobile sur 10 jours 
ca_journalier_b2b['mm_10j'] = ca_journalier_b2b['ca'].rolling(window=10).mean()
#moyenne mobile sur 30 jours 
ca_journalier_b2b['mm_30j'] = ca_journalier_b2b['ca'].rolling(window=30).mean()

print("moyenne mobiles calculées")
print(ca_journalier_b2b.head(10))

moyenne mobiles calculées
        date      ca       mm_7j   mm_10j  mm_30j
0 2021-03-01  222.79         NaN      NaN     NaN
1 2021-03-02  341.79         NaN      NaN     NaN
2 2021-03-03  302.16         NaN      NaN     NaN
3 2021-03-04  680.96         NaN      NaN     NaN
4 2021-03-05  539.96         NaN      NaN     NaN
5 2021-03-06  694.78         NaN      NaN     NaN
6 2021-03-07  476.96  465.628571      NaN     NaN
7 2021-03-08  390.33  489.562857      NaN     NaN
8 2021-03-09  504.96  512.872857      NaN     NaN
9 2021-03-10  342.38  518.618571  449.707     NaN


### Moyenne mobile B2C

In [60]:
#trier par date (inspensable pour les moy mobile)
ca_journalier_b2c = ca_journalier_b2c.sort_values('date').reset_index(drop=True)
#moyenne mobile sur 7 jours
ca_journalier_b2c['mm_7j'] = ca_journalier_b2c['ca'].rolling(window=7).mean()
#moyenne mobile sur 10 jours 
ca_journalier_b2c['mm_10j'] = ca_journalier_b2c['ca'].rolling(window=10).mean()
#moyenne mobile sur 30 jours 
ca_journalier_b2c['mm_30j'] = ca_journalier_b2c['ca'].rolling(window=30).mean()

print("moyenne mobiles calculées")
print(ca_journalier_b2c.head(10))

moyenne mobiles calculées
        date        ca         mm_7j     mm_10j  mm_30j
0 2021-03-01  16342.43           NaN        NaN     NaN
1 2021-03-02  15144.66           NaN        NaN     NaN
2 2021-03-03  14896.53           NaN        NaN     NaN
3 2021-03-04  14515.11           NaN        NaN     NaN
4 2021-03-05  16931.41           NaN        NaN     NaN
5 2021-03-06  15090.50           NaN        NaN     NaN
6 2021-03-07  14283.24  15314.840000        NaN     NaN
7 2021-03-08  15289.20  15164.378571        NaN     NaN
8 2021-03-09  15205.55  15173.077143        NaN     NaN
9 2021-03-10  15154.49  15209.928571  15285.312     NaN


In [61]:
#Graphique des moyenne mobile 

fig = make_subplots(rows=2, cols=1, subplot_titles=("Chiffre d'affaire B2B", "Chiffre d'affaire B2C"), vertical_spacing=0.15)

fig.add_trace(go.Scatter(
    x=ca_journalier_b2b['date'], 
    y=ca_journalier_b2b['ca'],
    mode='lines',
    name='CA Journalier ',
    line=dict(width=0.3, color='gray'),
    hovertemplate = "%{y:,.0f}€"
    ),
    row = 1, col = 1
)
fig.add_trace(go.Scatter(
    x=ca_journalier_b2b['date'], 
    y=ca_journalier_b2b['mm_7j'],
    mode='lines',
    name='Moyenne Mobile 7j',
    line=dict(width=2.5, color='blue'),
    hovertemplate = "%{y:,.0f}€"
    ),
    row = 1, col = 1
)

fig.add_trace(go.Scatter(
    x=ca_journalier_b2b['date'], 
    y=ca_journalier_b2b['mm_30j'],
    mode='lines',
    name='Moyenne Mobile 30j',
    line=dict(width=4.5, color='red'),
    hovertemplate = "%{y:,.0f}€"
    ),
    row = 1, col = 1
)

fig.add_trace(go.Scatter(
    x=ca_journalier_b2c['date'], 
    y=ca_journalier_b2c['ca'],
    mode='lines',
    name='CA Journalier B2C',
    line=dict(width=0.3, color='gray'),
    hovertemplate = "%{y:,.0f}€",
    showlegend=False
    ),
    row = 2, col = 1
)
fig.add_trace(go.Scatter(
    x=ca_journalier_b2c['date'], 
    y=ca_journalier_b2c['mm_7j'],
    mode='lines',
    name='Moyenne Mobile B2C 7j',
    line=dict(width=2.5, color='blue'),
    hovertemplate = "%{y:,.0f}€",
    showlegend=False
    ),
    row = 2, col = 1
)

fig.add_trace(go.Scatter(
    x=ca_journalier_b2c['date'], 
    y=ca_journalier_b2c['mm_30j'],
    mode='lines',
    name='Moyenne Mobile B2C 30j',
    line=dict(width=4.5, color='red'),
    hovertemplate = "%{y:,.0f}€",
    showlegend=False
    ),
    row = 2, col = 1
)
# mise en forme
fig.update_layout(
    title="Analyse du CA et Moyennes Mobiles",
    height = 700, 
    yaxis_title="Montant (€)",
    template="plotly_white",
    hovermode="x unified",
    separators = ". "
)

fig.update_xaxes(title_text="Date", row=2, col=1)  # axe X seulement du subplot du bas
fig.update_yaxes(title_text="Montant (€)", row=1, col=1)  # axe Y du subplot du haut
fig.update_yaxes(title_text="Montant (€)", row=2, col=1)  # axe Y du subplot du bas
fig.show()

### 2.1.1 tris du Dataframe en B2C

In [62]:
df_b2c = df_b2b_age[df_b2b_age["segment_client"]== "B2C"].copy()
df_b2b = df_b2b_age[df_b2b_age["segment_client"]== "B2B"].copy()

In [63]:
# 1. Vérification qu'il ne reste que du B2C
print(df_b2c['segment_client'].unique())


['B2C']


## 2.2 Chiffre d'affaire par catégories 

In [64]:
ca_categorie = df_b2b_age[df_b2b_age['segment_client'] == 'B2C'].groupby(by="categ")['price'].sum().reset_index()

ca_categorie.head()

,categ,price
0,0,4205283.73
1,1,4717565.67
2,2,2778773.81


In [65]:
fig = go.Figure()
fig.add_trace(go.Pie(
    labels =ca_categorie['categ'], 
    values =ca_categorie['price'],
    name='CA par catégories',
    marker = dict(colors = ['#4a1010', '#ac4f1f', '#ffa600']),
    hovertemplate = "%{value:,.0f}€",
    hole = 0.5
))
# mise en forme
fig.update_layout(
    title_text='Répartition du CA par catégorie',
    xaxis=dict(type='category'),
    separators = ". ",
    height = 500,
    width = 500
)

fig.show()

In [66]:
df_b2b_age.head()

,id_prod,date,session_id,client_id,price,categ,sex,birth,segment_client,age
0,0_1259,2021-03-01 00:01:07.843138,s_1,c_329,11.99,0,f,1967,B2C,54
1,0_1390,2021-03-01 00:02:26.047414,s_2,c_664,19.37,0,m,1960,B2C,61
2,0_1352,2021-03-01 00:02:38.311413,s_3,c_580,4.50,0,m,1988,B2C,33
3,0_1458,2021-03-01 00:04:54.559692,s_4,c_7912,6.55,0,f,1989,B2C,32
4,0_1358,2021-03-01 00:05:18.801198,s_5,c_2033,16.49,0,f,1956,B2C,65


## 2.3 Nombre de client par mois 

In [67]:
#grouper les clients uniques par mois
client_mois = df_b2c.groupby(pd.Grouper(key='date', freq='MS'))['client_id'].nunique().reset_index()
client_mois.rename(columns={"client_id":"client"}, inplace=True)
client_mois.head()

,date,client
0,2021-03-01,5675
1,2021-04-01,5673
2,2021-05-01,5643
3,2021-06-01,5658
4,2021-07-01,5671


In [68]:
#grouper les clients uniques par mois
client_mois_total = df_b2c.groupby(pd.Grouper(key='date', freq='MS'))['client_id'].count().reset_index()
client_mois_total.rename(columns={"client_id":"client"}, inplace=True)
client_mois_total.head()

,date,client
0,2021-03-01,27518
1,2021-04-01,27352
2,2021-05-01,27153
3,2021-06-01,25900
4,2021-07-01,23909


In [69]:
#grouper les clients totaux par mois
client_jeune  = df_b2c[df_b2c['age'] <= 20].groupby(pd.Grouper(key='date', freq='MS'))["client_id"].count().reset_index()
client_jeune.rename(columns={"client_id":"client"}, inplace=True)
client_jeune.head(30)

,date,client
0,2021-03-01,1111
1,2021-04-01,1138
2,2021-05-01,1161
3,2021-06-01,1207
4,2021-07-01,1344
5,2021-08-01,1252
6,2021-09-01,997
7,2021-10-01,1444
8,2021-11-01,1163
9,2021-12-01,1121


In [70]:
#definition du nombre de client de 20 ans ou moins 

client_jeune  = df_b2c[df_b2c['age'] <= 20].groupby(pd.Grouper(key='date', freq='MS'))["client_id"].count().reset_index()
client_jeune.rename(columns={"client_id":"client"}, inplace=True)
client_jeune.head(30)

,date,client
0,2021-03-01,1111
1,2021-04-01,1138
2,2021-05-01,1161
3,2021-06-01,1207
4,2021-07-01,1344
5,2021-08-01,1252
6,2021-09-01,997
7,2021-10-01,1444
8,2021-11-01,1163
9,2021-12-01,1121


In [87]:
#traçage des courbes
fig = make_subplots(rows=3, cols=1, subplot_titles=("Nombre de clients totaux par mois", "Nombre de clients unique par mois", "client de 20 ans ou moins"), vertical_spacing=0.18)

fig.add_trace(go.Scatter(
    x=client_mois['date'], 
    y=client_mois['client'],
    mode='lines',
    name='clients unique par mois',
    line=dict(width=2, color='#BF5F1E'),
),
    row = 2, col = 1
)

fig.add_trace(go.Scatter(
    x=client_mois_total['date'],
    y=client_mois_total['client'],
    mode='lines',
    name = 'clients totaux par mois',
    line=dict(width=2, color='#4A1010'),
),
    row = 1, col = 1
)

fig.add_trace(go.Scatter(
    x=client_jeune['date'],
    y=client_jeune['client'],
    mode='lines',
    name = 'clients moins de 20 ans',
    line=dict(width=2, color='#FFA600'),
),
    row = 3, col = 1
)

# mise en forme
fig.update_layout(
    title="Évolution des clients mensuel",
    xaxis_title="Mois",
    height = 800,
    showlegend = False,
    margin= dict(t=40, b=20, l=10, r=10)
)

fig.update_xaxes(hoverformat = "%b %Y", tickformat = "%B %Y")
fig.show()

## 2.4 Nombre de transaction par mois 

In [72]:
#grouper les transaction par jour/semaines/mois
transac_jour = df_b2c.groupby(pd.Grouper(key='date', freq='D'))['session_id'].nunique().reset_index()
transac_jour.rename(columns={"session_id":"Transaction"}, inplace=True)

transac_semaine = df_b2c.groupby(pd.Grouper(key='date', freq='W'))['session_id'].nunique().reset_index()
transac_semaine.rename(columns={"session_id":"Transaction"}, inplace=True)

transac_mois = df_b2c.groupby(pd.Grouper(key='date', freq='M'))['session_id'].nunique().reset_index()
transac_mois.rename(columns={"session_id":"Transaction"}, inplace=True)


C:\Users\quent\AppData\Local\Temp\ipykernel_12232\394331061.py:8: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  transac_mois = df_b2c.groupby(pd.Grouper(key='date', freq='M'))['session_id'].nunique().reset_index()


## 2.5 Nombre de produit vendu par mois

In [73]:
produit_mois = df_b2c.groupby(pd.Grouper(key='date', freq='M'))['id_prod'].nunique().reset_index()
produit_mois.rename(columns={"id_prod":"produit"}, inplace=True)


C:\Users\quent\AppData\Local\Temp\ipykernel_12232\2877287555.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  produit_mois = df_b2c.groupby(pd.Grouper(key='date', freq='M'))['id_prod'].nunique().reset_index()


In [74]:
#traçage des courbes
fig = make_subplots(rows=2, cols=1, subplot_titles=("Nombre de transactions uniques par mois", "Nombre de produits unique vendus par mois"), vertical_spacing=0.18)

fig.add_trace(go.Scatter(
    x=transac_mois['date'],
    y=transac_mois['Transaction'],
    mode='lines',
    name = 'Transaction par mois',
    line=dict(width=2, color='orange'),
),
    row = 1, col = 1
)

fig.add_trace(go.Scatter(
    x=produit_mois['date'],
    y=produit_mois['produit'],
    mode='lines',
    name = 'Produits vendus',
    line=dict(width=2, color='green'),
    ),
    row = 2, col = 1
)


# mise en forme
fig.update_layout(
    title="Évolution des clients mensuel",
    xaxis_title="Mois",
    height = 600,
    showlegend = False,
    margin= dict(t=40, b=20, l=10, r=10),
    hovermode="x",
    hoverlabel=dict(
    bgcolor="white",    # couleur de fond
    font_size=12,       # taille de la police
    font_family="Arial" # police
)
)

fig.update_xaxes(hoverformat = "%b %Y", tickformat = "%B %Y")
fig.show()

In [75]:
session_unique = df_b2c['session_id'].nunique()
nombre_session = df_b2c['session_id'].count()
diffrence = nombre_session - session_unique
print(f'nombre de session unique {session_unique}')
print(f'le nombre total de session {nombre_session}')
print(f'il ya {diffrence} session non unique ')

nombre de session unique 334508
le nombre total de session 661948
il ya 327440 session non unique 


In [76]:
#ca mensuel par categorie 

ca_mensuel_categ = df_b2c.groupby([pd.Grouper(key='date', freq='M'), 'categ'])['price'].sum().reset_index()
ca_mensuel_categ.head()

C:\Users\quent\AppData\Local\Temp\ipykernel_12232\427659552.py:3: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  ca_mensuel_categ = df_b2c.groupby([pd.Grouper(key='date', freq='M'), 'categ'])['price'].sum().reset_index()


,date,categ,price
0,2021-03-31,0,184153.37
1,2021-03-31,1,182709.30
2,2021-03-31,2,101837.27
3,2021-04-30,0,195725.91
4,2021-04-30,1,152091.56


In [77]:
# creation de la courbe
fig = go.Figure()


for c in ca_mensuel_categ['categ'].unique():
    # On isole les données de la catégorie
    data_filtree = ca_mensuel_categ[ca_mensuel_categ['categ'] == c]
    
    fig.add_trace(go.Scatter(
        x=data_filtree['date'],
        y=data_filtree['price'],
        mode='lines', 
        name=f"Catégorie {c}",
        connectgaps=True, # Au cas où un mois n'aurait pas de données
        hovertemplate = "%{y:.0f}€"
    )
    )


# mise en forme
fig.update_layout(
    title="Évolution du CA mensuel par catégorie",
    xaxis_title="Mois",
    yaxis_title="CA (€)",
    hovermode="x unified" # affiche toutes les valeurs au même point X au survol
    
)

fig.show()

## 2.6 Les top/flop

In [78]:
#tris en top 5 des produit les plus vendus
df_top5 = df_b2c.groupby('id_prod')['price'].sum().sort_values(ascending=False).head(5).reset_index()


In [79]:
#tris en flop 5 des produit les plus vendus
df_flop5 = df_b2c.groupby('id_prod')['price'].sum().sort_values(ascending=True).head(5).reset_index()

In [88]:
#Traçage du bar plot
fig = make_subplots(rows=1, cols=2, subplot_titles=("Top 5 des produits vendus", "Flop 5 des produits vendus"), horizontal_spacing=0.05)

fig.add_trace(go.Bar(
    x=df_top5['id_prod'].astype(str),
    y=df_top5['price'],
    name="top 5 produits vendus",
    marker_color = ["#4a1010","#7c2c1b", "#ac4f1f","#d97819", "#ffa600" ],
    hovertemplate = "%{y:.0f}€<extra></extra>"
),
row=1, col= 1,
)

fig.add_trace(go.Bar(
    x=df_flop5['id_prod'].astype(str),
    y=df_flop5['price'],
    name="flop 5 produits vendus",
    marker_color = ["#132d7a","#762b84", "#b92779","#ea3e5f", "#ff6f3c" ],
    hovertemplate = "%{y:.0f}€<extra></extra>"
),
row=1, col= 2,
)
fig.update_layout(
    showlegend = False,
    title = "Top 10 des produits par CA (B2C)",
    xaxis_title = "ID Produit",
    yaxis_title = "Chiffre d'Affaires (€)",
    xaxis={'categoryorder':'total descending'} # Force l'ordre décroissant visuel


)
fig.show()

## 2.7 les Flops des produit vendus 

In [81]:
# tri du flop 5 des produit 

df_flop10 = df_b2c.groupby('id_prod')['price'].sum().sort_values(ascending=True).head(5).reset_index()

In [82]:
#Traçage du bar plot
fig = go.Figure()
    
fig.add_trace(go.Bar(
    x=df_flop10['id_prod'].astype(str),
    y=df_flop10['price'],
    name="flop 5 produit vendu",
    marker_color = ["#4a1010","#7c2c1b", "#ac4f1f","#d97819", "#ffa600" ],
    hovertemplate = "%{y:.0f}€<extra></extra>"
))
fig.update_layout(
    title = "flop 5 des produits par CA (B2C)",
    xaxis_title = "ID Produit",
    yaxis_title = "Chiffre d'Affaires (€)",
    xaxis={'categoryorder':'total descending'} # Force l'ordre décroissant visuel
)
fig.show()

In [83]:
nb_categorie = df_b2c.groupby('categ')['id_prod'].nunique().reset_index()

In [84]:
fig = go.Figure()
fig.add_trace(go.Pie(
    labels =nb_categorie['categ'], 
    values =nb_categorie['id_prod'],
    name='produits par catégories',
    marker = dict(colors = ['#4a1010', '#ac4f1f', '#ffa600']),
    #textinfo="label",
    hole = 0.5
))
# mise en forme
fig.update_layout(
    title_text='Répartition des produits par catégorie',
    xaxis=dict(type='category'),
    separators = ". ",
    height = 500,
    width = 500
)

fig.show()

# 3.Vision client 

## 3.1 Répartition du CA des clients B2B  